# Panda power three bus problem.

In [ ]:
import pandapower as pp

# Create empty net
# https://pandapower.readthedocs.io/en/stable/elements/empty_network.html

net = pp.create_empty_network() 


In [ ]:
# Create 3 buses, b1,b2,b3
# https://pandapower.readthedocs.io/en/stable/elements/bus.html
V_nom_kV = 220

b1 = pp.create_bus(net, vn_kv=V_nom_kV,  name="Bus 1")
b2 = # TODO


# inspect the bus objects
net.bus

In [ ]:
###### create bus elements

# BUS 1 
# Create a swing (also known as a slack) bus, V=1.05 py Angle =0
# Note: Its called an external grid in PP
# https://pandapower.readthedocs.io/en/stable/elements/ext_grid.html

pp.create_ext_grid(net, bus=b1, vm_pu=1.05, name="Swing bus")


# BUS 2 
# Create a load bus, P=256.6MW, Q=110.2 Mvar
# https://pandapower.readthedocs.io/en/stable/elements/load.html
# TODO


# BUS 3 
# Create a load bus, P=138.6MW, Q=45.2 Mvar
# https://pandapower.readthedocs.io/en/stable/elements/load.html
# TODO

# inspect the load objects
net.load


In [ ]:
## Create lines
#https://pandapower.readthedocs.io/en/stable/elements/line.html
# NOTE: if x,r in PU then set length_km=Z_base to get from PU values to actual values.

# Calculate Z_Base = V/I = V^2/VI
VA_nom_MVA = 100 
Z_base = (V_nom_kV)**2/VA_nom_MVA
print ("Z_base = ",Z_base)

# Line 1-2 0.02+j0.04 
pp.create_line_from_parameters(
    net, from_bus=b1, to_bus=b2,length_km=Z_base,
    r_ohm_per_km = 0.02, x_ohm_per_km = 0.04,
    c_nf_per_km = 1e-9, max_i_ka=1000
    )

# Line 2-3 0.0125+j0.025 
# TODO


# Line 1-3 0.01+j0.03 
# TODO



In [ ]:
# inspect the line objects
net.line

In [ ]:
# Do a power flow
# https://pandapower.readthedocs.io/en/stable/powerflow/ac.html

pp.runpp(net, algorithm='gs');

In [ ]:
# Save the network to a json file
# https://pandapower.readthedocs.io/en/stable/file_io.html

pp.to_json(net,"ThreeBus.json")

In [ ]:
# View results on the buses
# TODO

In [ ]:
# View results on the lines
# TODO

How do the results compare to your Gauss-Seidel solution from D6?

## Plotting
Pandapower also provides the ability to plot the network - including line loadings etc. 
Use the ```pplot.simple_plot``` to view a schematic of the network (with arbitrary locations assigned).

In [ ]:
# Simple plotting
# https://pandapower.readthedocs.io/en/stable/plotting.html
import pandapower.plotting as pplot

# TODO

Now add the following geospatial data to the buses:

| Bus  | Longitude (x) | Latitude (y) |
|-----:|--------------:|-------------:|
| Bus 1 | -1.2763 | 51.7503   |
| Bus 2 |  -1.189      | 51.7236      |
| Bus 3 | -1.3269 | 51.6312  |

You can either edit the code above when you created the buses in the first place (and re-run the notebook), or you can edit the bus dataframe directly. Pandapower stores the geospatial data in geojson format within the ```geo``` column. The code below shows you how to create geodataframes for the buses and lines, and how to assign the geospatial data to the buses. Pandapower uses the WGS84 coordinate system (EPSG:4326) for geospatial data (e.g. longitude and latitude).

In [ ]:
import geopandas as gpd
import json
from shapely.geometry import Point, LineString, mapping

# -----------------------------------
# 1. Create bus GeoDataFrame
# -----------------------------------

bus_coords = {
    0: (-1.2763, 51.7503),
    1: (-1.1890, 51.7236),
    2: (-1.3269, 51.6312)
}

gdf_buses = gpd.GeoDataFrame(
    geometry=[Point(coords) for coords in bus_coords.values()],
    index=list(bus_coords.keys()),
    crs="EPSG:4326"
)

# -----------------------------------
# 2. Create line GeoDataFrame
# notice lines are lists of coordinates, here we assume a straight line between the buses, but you could also use a more complex routing geometries by providing more points.
# -----------------------------------

line_coords = {
    0: [bus_coords[0], bus_coords[1]],
    1: [bus_coords[1], bus_coords[2]],
    2: [bus_coords[0], bus_coords[2]]
}

gdf_lines = gpd.GeoDataFrame(
    geometry=[LineString(coords) for coords in line_coords.values()],
    index=list(line_coords.keys()),
    crs="EPSG:4326"
)

# -----------------------------------
# 3. Assign geometries to pandapower
# -----------------------------------

net.bus.loc[gdf_buses.index, "geo"] = gdf_buses.geometry.apply(
    lambda geom: json.dumps(mapping(geom))
)

net.line.loc[gdf_lines.index, "geo"] = gdf_lines.geometry.apply(
    lambda geom: json.dumps(mapping(geom))
)


## Plotting with plotly
Pandapower has added support for plotly plotting. This allows for interactive plots, including zooming and panning. Use the ```pplot.simple_plotly``` function to view the network in an interactive plot. You can also plot the power flow results using the pf_res_plotly function. 

Note: There appears to be a bug in the current version of the pandapower plotting library related to a reverse geolocated trying to assign an address to points. The code below bypasses this issue.

The documentation is here: [pandapower plotly plotting](https://pandapower.readthedocs.io/en/stable/plotting.html#plotting-with-plotly)

In [ ]:
import pandapower.plotting.plotly.traces as traces

# Temporary workaround for reverse-geocoding failure
traces._on_map_test = lambda x, y: True

fig = pp_plot.simple_plotly(
    net,
    on_map=True,
    use_line_geo=True,
    map_style="open-street-map",
    aspectratio=(1,1), figsize=1
    
)

In [ ]:
# TODO: Re-plot the data, including color coding of the power flow results using the pf_res_plotly function.



## Exporting results to a geodataframe
pandapower has removed support for direct export of results to a geodataframe. You can use the code below to create geodataframes.

In [ ]:
import geopandas as gpd
# create a GeoDataFrame
bus_geojson = pp_plot.geo.dump_to_geojson(net, buses=True)
bus_gdf = gpd.GeoDataFrame.from_features(bus_geojson["features"], crs="EPSG:4326")
lines_geojson = pp_plot.geo.dump_to_geojson(net, lines=True)
lines_gdf = gpd.GeoDataFrame.from_features(lines_geojson["features"], crs="EPSG:4326")

You can then perform spatial analysis on the results if necessary. 


In [ ]:
lines.gdf.length_km = lines_gdf.geometry.length * 100